In this notebook, we check if data quality metrics actually correlate with a teacher's performance on some benchmarks like Global-MMLU and M-GSM (and maybe M-RewardBench). For now, we'll only investigate the following languages: de, es, ja because they are present in these benchmarks.

In [ ]:
import json
import pandas as pd
from typing import Any
from datasets import Dataset, DownloadMode, load_dataset
from huggingface_hub import list_datasets

In [ ]:
datasets = [
    ds.id
    for ds in list_datasets(search="details_", author="ljvmiranda921")
    if "quant" not in ds.id
]
print(f"Found {len(datasets)} datasets")
print(datasets)


def parse_outputs(dataset_id: str) -> dict[str, Any]:
    """Parse a dataset ID and output a dataframe containing the relevant fields

    Based from: https://huggingface.co/docs/lighteval/en/saving-and-reading-results
    """
    print(f"Parsing results from dataset {dataset_id}")
    ds = load_dataset(
        dataset_id,
        "results",
        trust_remote_code=True,
        download_mode=DownloadMode.FORCE_REDOWNLOAD,
    )

    # Save all metrics and versions for each task
    metrics = {}
    versions = {}
    for run in ds.keys():
        df = ds[run].to_pandas()
        for task, result in json.loads(df.results.iloc[0]).items():
            if task != "all":
                try:
                    _, benchmark, n_shots = task.split("|")
                except ValueError:
                    benchmark, n_shots = task.split("|")
                if (
                    "global_mmlu_lite" in benchmark
                    or "mgsm_custom" in benchmark
                    or "mrewardbench" in benchmark
                ):
                    metrics[benchmark] = result

        versions.update(json.loads(df.versions.iloc[0]))

    print(f"Found {len(metrics)} tasks!")

    latest_config = json.loads(ds["latest"].to_pandas().config_general.iloc[0])
    model_config = {
        "model_name": latest_config.get("model_name"),
        "model_dtype": latest_config.get("model_dtype"),
        "model_size": latest_config.get("model_size"),
    }

    return {
        "config": model_config,
        "results": metrics,
        "versions": versions,
    }

In [ ]:
parsed_results = pd.DataFrame([parse_outputs(dataset) for dataset in datasets])

In [ ]:
parsed_results.iloc[0].results

In [ ]:
parsed_results

Now let's analyze if there's a correlation between teacher model performance on Global-MMLU Lite and the quality of synthetic data they generate for German, Japanese, and Spanish.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

sys.path.append("../../scripts/")
from analysis.model_info import MODEL_INFORMATION
from analysis import plot_theme

# Load plot parameters
plot_params = plot_theme.PLOT_PARAMS.copy()
plt.rcParams.update(plot_params)

In [ ]:
# Load data quality metrics
df_quality = pd.read_json(
    "../../notebooks/2025-11-24-data-quality-metrics/data/compiled_intrinsic_metrics.jsonl",
    orient="records",
    lines=True,
)


def calculate_quality_score(df):
    """Calculate an aggregate quality score from intrinsic metrics using z-score normalization.

    Each metric is standardized (mean=0, std=1) and then averaged. This approach:
    - Handles extreme outliers automatically
    - Gives equal weight to each metric in terms of variance
    - Incorporates all 4 metrics: prompts_distinct_ri, responses_distinct_ri,
      rubric_score, and perplexity (log-transformed and inverted)
    """
    data = np.column_stack(
        [
            df["prompts_distinct_ri"],
            df["responses_distinct_ri"],
            df["rubric_score"],
            -np.log1p(df["perplexity"]),
        ]
    )

    # Standardize and average
    scaler = StandardScaler()
    normalized = scaler.fit_transform(data)
    quality_score = normalized.mean(axis=1)

    return quality_score


# Calculate quality score
df_quality["quality_score"] = calculate_quality_score(df_quality)

# Filter for languages we're interested in
df_quality_filtered = df_quality[df_quality["language"].isin(["de", "es", "ja"])].copy()

df_quality_filtered.head()

In [ ]:
# Extract MMLU-Lite scores from MODEL_INFORMATION
mmlu_scores = []

for model_info in MODEL_INFORMATION:
    if model_info.benchmark_scores:
        for benchmark_key, scores in model_info.benchmark_scores.items():
            if "global_mmlu_lite" in benchmark_key:
                # Extract language from benchmark key (e.g., "global_mmlu_lite:de" -> "de")
                lang = benchmark_key.split(":")[-1]
                mmlu_scores.append(
                    {
                        "model": model_info.name,
                        "language": lang,
                        "mmlu_acc": scores["acc"],
                        "mmlu_stderr": scores["acc_stderr"],
                    }
                )

df_mmlu = pd.DataFrame(mmlu_scores)

# Merge MMLU scores with quality metrics
df_merged = df_quality_filtered.merge(df_mmlu, on=["model", "language"], how="inner")

print(f"Merged {len(df_merged)} model-language pairs")
df_merged[["model", "language", "mmlu_acc", "quality_score"]].sort_values(
    "mmlu_acc", ascending=False
)

In [ ]:
# Calculate correlation for each language
from scipy.stats import pearsonr, spearmanr

correlations = {}
for lang in ["de", "es", "ja"]:
    df_lang = df_merged[df_merged["language"] == lang]

    pearson_corr, pearson_p = pearsonr(df_lang["mmlu_acc"], df_lang["quality_score"])
    spearman_corr, spearman_p = spearmanr(df_lang["mmlu_acc"], df_lang["quality_score"])

    correlations[lang] = {
        "pearson": pearson_corr,
        "pearson_p": pearson_p,
        "spearman": spearman_corr,
        "spearman_p": spearman_p,
    }

In [ ]:
# Create scatter plots for each language
from analysis.plot_theme import COLORS
from pathlib import Path
from matplotlib.offsetbox import OffsetImage, AnnotationBbox, HPacker, TextArea
from PIL import Image

# Create figures directory if it doesn't exist
figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)

# Create logos directory
logos_dir = Path("logos")


# Load logo helper function
def load_logo_image(filepath, target_size=30):
    """Load a PNG logo from file and resize to fixed pixel size"""
    try:
        img = Image.open(filepath)

        # Convert to RGBA to preserve transparency
        if img.mode != "RGBA":
            img = img.convert("RGBA")

        # Resize maintaining aspect ratio
        img.thumbnail((target_size, target_size), Image.Resampling.LANCZOS)

        return OffsetImage(img, zoom=0.8)
    except Exception as e:
        print(f"Failed to load {filepath}: {e}")
        return None


# Map families to logo files
family_logo_map = {
    "Aya": logos_dir / "aya.png",
    "Gemma": logos_dir / "google.png",
    "Granite": logos_dir / "ibm.png",
    "Llama": logos_dir / "meta.png",
}

# Pre-load all logos
print("Loading model family logos...")
logo_images = {}
for family, filepath in family_logo_map.items():
    if filepath.exists():
        img = load_logo_image(filepath, target_size=30)
        if img:
            logo_images[family] = img
            print(f"✓ Loaded {family} logo from {filepath}")
    else:
        print(f"✗ Logo not found: {filepath}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

language_names = {
    "de": "German",
    "es": "Spanish",
    "ja": "Japanese",
}

languages = ["de", "es", "ja"]

for idx, (lang, ax) in enumerate(zip(languages, axes)):
    df_lang = df_merged[df_merged["language"] == lang].copy()

    # Add model family information for coloring
    model_info_df = pd.DataFrame([m.model_dump() for m in MODEL_INFORMATION])
    df_lang = df_lang.merge(
        model_info_df[["name", "model_family"]],
        left_on="model",
        right_on="name",
        how="left",
    )

    # Define colors for model families
    family_colors = {
        "Aya": COLORS["warm_blue"],
        "Gemma": COLORS["warm_crest"],
        "Granite": COLORS["heritage"],
        "Llama": COLORS["slate"],
    }

    # Plot each model family with logos
    for family in df_lang["model_family"].unique():
        family_data = df_lang[df_lang["model_family"] == family]

        for _, row in family_data.iterrows():
            x_pos = row["mmlu_acc"]
            y_pos = row["quality_score"]

            ab = AnnotationBbox(
                logo_images[family],
                (x_pos, y_pos),
                xycoords="data",
                frameon=False,
                box_alignment=(0.5, 0.5),
                zorder=3,
                pad=0,
            )
            ax.add_artist(ab)

    # Add linear regression line
    from scipy.stats import linregress

    slope, intercept, r_value, p_value, std_err = linregress(
        df_lang["mmlu_acc"], df_lang["quality_score"]
    )
    x_line = np.linspace(df_lang["mmlu_acc"].min(), df_lang["mmlu_acc"].max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, "r--", alpha=0.5, linewidth=2, zorder=2)

    # Add correlation info to title
    pearson_corr = correlations[lang]["pearson"]
    spearman_corr = correlations[lang]["spearman"]
    ax.set_title(
        f"{language_names[lang]}\n$r={pearson_corr:.2f}$, $\\rho={spearman_corr:.2f}$"
    )

    ax.set_xlabel("Global MMLU-Lite Acc.")
    if idx == 0:
        ax.set_ylabel("Data Quality Score")

    ax.grid(True, alpha=0.3, zorder=1)

# Create custom legend with logos at the bottom
legend_families = ["Aya", "Gemma", "Granite", "Llama"]

for family in legend_families:
    if family in logo_images:
        # Load a bigger version for legend
        legend_img = load_logo_image(family_logo_map[family], target_size=40)
        if legend_img:
            # Create a text area with logo for legend
            text_box = TextArea(family, textprops=dict(size=18))
            h_packer = HPacker(
                children=[legend_img, text_box], align="center", pad=0, sep=8
            )

            # Add to bottom center - moved lower with -0.20
            ab = AnnotationBbox(
                h_packer,
                (-0.2 + 0.5 * legend_families.index(family), -0.40),
                xycoords="axes fraction",
                frameon=False,
                box_alignment=(0.5, 0.5),
            )
            axes[1].add_artist(ab)

# plt.tight_layout()
fig.savefig(
    figures_dir / "mmlu_vs_quality_correlation.png", dpi=300, bbox_inches="tight"
)

plt.show()

Now let's check the correlation between MRewardBench performance and data quality scores.

In [ ]:
mrewardbench_scores = []

# Models to exclude (incomplete results)
exclude_models = [
    "meta-llama/Llama-3.1-70B-Instruct",
    "mistralai/Mistral-Small-24B-Instruct-2501",
]

for model_info in MODEL_INFORMATION:
    if model_info.name in exclude_models:
        continue
    if model_info.benchmark_scores:
        for benchmark_key, scores in model_info.benchmark_scores.items():
            if "mrewardbench_mcf" in benchmark_key:
                lang = benchmark_key.split(":")[-1]
                mrewardbench_scores.append(
                    {
                        "model": model_info.name,
                        "language": lang,
                        "mrewardbench_acc": scores["weighted_acc"],
                        "mrewardbench_stderr": scores["weighted_acc_stderr"],
                    }
                )

df_mrewardbench = pd.DataFrame(mrewardbench_scores)
df_merged_mrb = df_quality_filtered.merge(
    df_mrewardbench, on=["model", "language"], how="inner"
)

print(f"Merged {len(df_merged_mrb)} model-language pairs for MRewardBench")
df_merged_mrb[["model", "language", "mrewardbench_acc", "quality_score"]].sort_values(
    "mrewardbench_acc", ascending=False
).head(10)

In [ ]:
# Calculate correlation for MRewardBench
correlations_mrb = {}
for lang in ["de", "es", "ja"]:
    df_lang = df_merged_mrb[df_merged_mrb["language"] == lang]

    pearson_corr, pearson_p = pearsonr(
        df_lang["mrewardbench_acc"], df_lang["quality_score"]
    )
    spearman_corr, spearman_p = spearmanr(
        df_lang["mrewardbench_acc"], df_lang["quality_score"]
    )

    correlations_mrb[lang] = {
        "pearson": pearson_corr,
        "pearson_p": pearson_p,
        "spearman": spearman_corr,
        "spearman_p": spearman_p,
    }

# Create scatter plots for MRewardBench
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, (lang, ax) in enumerate(zip(languages, axes)):
    df_lang = df_merged_mrb[df_merged_mrb["language"] == lang].copy()

    # Add model family information
    model_info_df = pd.DataFrame([m.model_dump() for m in MODEL_INFORMATION])
    df_lang = df_lang.merge(
        model_info_df[["name", "model_family"]],
        left_on="model",
        right_on="name",
        how="left",
    )

    # Plot each model family with logos
    for family in df_lang["model_family"].unique():
        family_data = df_lang[df_lang["model_family"] == family]

        for _, row in family_data.iterrows():
            x_pos = row["mrewardbench_acc"]
            y_pos = row["quality_score"]

            # Create a fresh logo image for each plot point
            logo_img = load_logo_image(family_logo_map[family], target_size=30)
            if logo_img:
                ab = AnnotationBbox(
                    logo_img,
                    (x_pos, y_pos),
                    xycoords="data",
                    frameon=False,
                    box_alignment=(0.5, 0.5),
                    zorder=3,
                    pad=0,
                )
                ax.add_artist(ab)

    # Add linear regression line
    slope, intercept, r_value, p_value, std_err = linregress(
        df_lang["mrewardbench_acc"], df_lang["quality_score"]
    )
    x_line = np.linspace(
        df_lang["mrewardbench_acc"].min(), df_lang["mrewardbench_acc"].max(), 100
    )
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, "r--", alpha=0.5, linewidth=2, zorder=2)

    # Add correlation info to title
    pearson_corr = correlations_mrb[lang]["pearson"]
    spearman_corr = correlations_mrb[lang]["spearman"]
    ax.set_title(
        f"{language_names[lang]}\n$r={pearson_corr:.2f}$, $\\rho={spearman_corr:.2f}$"
    )

    ax.set_xlabel("MRewardBench Weighted Acc.")
    if idx == 0:
        ax.set_ylabel("Data Quality Score")

    ax.grid(True, alpha=0.3, zorder=1)

# Create custom legend with logos at the bottom
for family in legend_families:
    if family in family_logo_map and family_logo_map[family].exists():
        legend_img = load_logo_image(family_logo_map[family], target_size=40)
        if legend_img:
            text_box = TextArea(family, textprops=dict(size=18))
            h_packer = HPacker(
                children=[legend_img, text_box], align="center", pad=0, sep=8
            )

            ab = AnnotationBbox(
                h_packer,
                (-0.2 + 0.5 * legend_families.index(family), -0.40),
                xycoords="axes fraction",
                frameon=False,
                box_alignment=(0.5, 0.5),
            )
            axes[1].add_artist(ab)

fig.savefig(
    figures_dir / "mrewardbench_vs_quality_correlation.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Extract M-GSM scores from MODEL_INFORMATION (only models with complete results)
mgsm_scores = []

for model_info in MODEL_INFORMATION:
    if model_info.name in exclude_models:
        continue
    if model_info.benchmark_scores:
        for benchmark_key, scores in model_info.benchmark_scores.items():
            if "mgsm_custom" in benchmark_key:
                # Extract language from benchmark key
                lang = benchmark_key.split(":")[-1]
                mgsm_scores.append(
                    {
                        "model": model_info.name,
                        "language": lang,
                        "mgsm_acc": scores["extractive_match"],
                        "mgsm_stderr": scores["extractive_match_stderr"],
                    }
                )

df_mgsm = pd.DataFrame(mgsm_scores)

# Merge M-GSM scores with quality metrics
df_merged_mgsm = df_quality_filtered.merge(
    df_mgsm, on=["model", "language"], how="inner"
)

print(f"Merged {len(df_merged_mgsm)} model-language pairs for M-GSM")
df_merged_mgsm[["model", "language", "mgsm_acc", "quality_score"]].sort_values(
    "mgsm_acc", ascending=False
).head(10)

Finally, let's analyze the correlation between M-GSM (multilingual grade school math) performance and data quality scores.

In [ ]:
# Calculate correlation for M-GSM
correlations_mgsm = {}
for lang in ["de", "es", "ja"]:
    df_lang = df_merged_mgsm[df_merged_mgsm["language"] == lang]

    pearson_corr, pearson_p = pearsonr(df_lang["mgsm_acc"], df_lang["quality_score"])
    spearman_corr, spearman_p = spearmanr(df_lang["mgsm_acc"], df_lang["quality_score"])

    correlations_mgsm[lang] = {
        "pearson": pearson_corr,
        "pearson_p": pearson_p,
        "spearman": spearman_corr,
        "spearman_p": spearman_p,
    }

# Create scatter plots for M-GSM
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, (lang, ax) in enumerate(zip(languages, axes)):
    df_lang = df_merged_mgsm[df_merged_mgsm["language"] == lang].copy()

    # Add model family information
    model_info_df = pd.DataFrame([m.model_dump() for m in MODEL_INFORMATION])
    df_lang = df_lang.merge(
        model_info_df[["name", "model_family"]],
        left_on="model",
        right_on="name",
        how="left",
    )

    # Plot each model family with logos
    for family in df_lang["model_family"].unique():
        family_data = df_lang[df_lang["model_family"] == family]

        for _, row in family_data.iterrows():
            x_pos = row["mgsm_acc"]
            y_pos = row["quality_score"]

            # Create a fresh logo image for each plot point
            logo_img = load_logo_image(family_logo_map[family], target_size=30)
            if logo_img:
                ab = AnnotationBbox(
                    logo_img,
                    (x_pos, y_pos),
                    xycoords="data",
                    frameon=False,
                    box_alignment=(0.5, 0.5),
                    zorder=3,
                    pad=0,
                )
                ax.add_artist(ab)

    # Add linear regression line
    slope, intercept, r_value, p_value, std_err = linregress(
        df_lang["mgsm_acc"], df_lang["quality_score"]
    )
    x_line = np.linspace(df_lang["mgsm_acc"].min(), df_lang["mgsm_acc"].max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, "r--", alpha=0.5, linewidth=2, zorder=2)

    # Add correlation info to title
    pearson_corr = correlations_mgsm[lang]["pearson"]
    spearman_corr = correlations_mgsm[lang]["spearman"]
    ax.set_title(
        f"{language_names[lang]}\n$r={pearson_corr:.2f}$, $\\rho={spearman_corr:.2f}$"
    )

    ax.set_xlabel("M-GSM Extractive Match")
    if idx == 0:
        ax.set_ylabel("Data Quality Score")

    ax.grid(True, alpha=0.3, zorder=1)

# Create custom legend with logos at the bottom
for family in legend_families:
    if family in family_logo_map and family_logo_map[family].exists():
        legend_img = load_logo_image(family_logo_map[family], target_size=40)
        if legend_img:
            text_box = TextArea(family, textprops=dict(size=18))
            h_packer = HPacker(
                children=[legend_img, text_box], align="center", pad=0, sep=8
            )

            ab = AnnotationBbox(
                h_packer,
                (-0.2 + 0.5 * legend_families.index(family), -0.40),
                xycoords="axes fraction",
                frameon=False,
                box_alignment=(0.5, 0.5),
            )
            axes[1].add_artist(ab)

fig.savefig(
    figures_dir / "mgsm_vs_quality_correlation.png", dpi=300, bbox_inches="tight"
)

plt.show()

Finally, let's create an aggregate view by averaging performance across all three benchmarks (Global MMLU-Lite, M-GSM, and MRewardBench).

In [ ]:
# Merge all three benchmark dataframes to get complete data
df_combined = df_merged_mrb.copy()
df_combined = df_combined.merge(
    df_merged_mgsm[["model", "language", "mgsm_acc"]],
    on=["model", "language"],
    how="inner",
)
df_combined = df_combined.merge(
    df_merged[["model", "language", "mmlu_acc"]], on=["model", "language"], how="inner"
)

# Calculate average performance across all three benchmarks
# Note: We need to normalize them first since they're on different scales
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_combined["mmlu_acc_norm"] = scaler.fit_transform(df_combined[["mmlu_acc"]])
df_combined["mgsm_acc_norm"] = scaler.fit_transform(df_combined[["mgsm_acc"]])
df_combined["mrewardbench_acc_norm"] = scaler.fit_transform(
    df_combined[["mrewardbench_acc"]]
)

# Calculate average normalized performance
df_combined["avg_multilingual_perf"] = (
    df_combined["mmlu_acc_norm"]
    + df_combined["mgsm_acc_norm"]
    + df_combined["mrewardbench_acc_norm"]
) / 3

print(f"Combined dataset has {len(df_combined)} model-language pairs")
df_combined[
    ["model", "language", "avg_multilingual_perf", "quality_score"]
].sort_values("avg_multilingual_perf", ascending=False).head(10)

In [ ]:
# Calculate correlation for combined performance
correlations_combined = {}
for lang in ["de", "es", "ja"]:
    df_lang = df_combined[df_combined["language"] == lang]

    pearson_corr, pearson_p = pearsonr(
        df_lang["avg_multilingual_perf"], df_lang["quality_score"]
    )
    spearman_corr, spearman_p = spearmanr(
        df_lang["avg_multilingual_perf"], df_lang["quality_score"]
    )

    correlations_combined[lang] = {
        "pearson": pearson_corr,
        "pearson_p": pearson_p,
        "spearman": spearman_corr,
        "spearman_p": spearman_p,
    }

# Create scatter plots for combined performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, (lang, ax) in enumerate(zip(languages, axes)):
    df_lang = df_combined[df_combined["language"] == lang].copy()

    # Add model family information
    model_info_df = pd.DataFrame([m.model_dump() for m in MODEL_INFORMATION])
    df_lang = df_lang.merge(
        model_info_df[["name", "model_family"]],
        left_on="model",
        right_on="name",
        how="left",
    )

    # Plot each model family with logos
    for family in df_lang["model_family"].unique():
        family_data = df_lang[df_lang["model_family"] == family]

        for _, row in family_data.iterrows():
            x_pos = row["avg_multilingual_perf"]
            y_pos = row["quality_score"]

            # Create a fresh logo image for each plot point
            logo_img = load_logo_image(family_logo_map[family], target_size=30)
            if logo_img:
                ab = AnnotationBbox(
                    logo_img,
                    (x_pos, y_pos),
                    xycoords="data",
                    frameon=False,
                    box_alignment=(0.5, 0.5),
                    zorder=3,
                    pad=0,
                )
                ax.add_artist(ab)

    # Add linear regression line
    slope, intercept, r_value, p_value, std_err = linregress(
        df_lang["avg_multilingual_perf"], df_lang["quality_score"]
    )
    x_line = np.linspace(
        df_lang["avg_multilingual_perf"].min(),
        df_lang["avg_multilingual_perf"].max(),
        100,
    )
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, "r--", alpha=0.5, linewidth=2, zorder=2)

    # Add correlation info to title
    pearson_corr = correlations_combined[lang]["pearson"]
    spearman_corr = correlations_combined[lang]["spearman"]
    ax.set_title(
        f"{language_names[lang]}\n$r={pearson_corr:.2f}$, $\\rho={spearman_corr:.2f}$"
    )

    ax.set_xlabel("Multilingual Perf.")
    if idx == 0:
        ax.set_ylabel("Data Quality Score")

    ax.grid(True, alpha=0.3, zorder=1)

# Create custom legend with logos at the bottom
for family in legend_families:
    if family in family_logo_map and family_logo_map[family].exists():
        legend_img = load_logo_image(family_logo_map[family], target_size=40)
        if legend_img:
            text_box = TextArea(family, textprops=dict(size=18))
            h_packer = HPacker(
                children=[legend_img, text_box], align="center", pad=0, sep=8
            )

            ab = AnnotationBbox(
                h_packer,
                (-0.2 + 0.5 * legend_families.index(family), -0.40),
                xycoords="axes fraction",
                frameon=False,
                box_alignment=(0.5, 0.5),
            )
            axes[1].add_artist(ab)

fig.savefig(
    figures_dir / "combined_multilingual_vs_quality_correlation.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()